In [120]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
import tensorflow.keras as keras
from sklearn.preprocessing import StandardScaler
import time
from scipy import stats

from pathlib import Path

# --- ONE set of path definitions, pointing at Drive ---
DATA_DIR = Path("/content/drive/MyDrive/AMEX1B - Data ")  # trailing space is part of the real folder name
CSV_DIR = DATA_DIR / "csv_files"
CACHE_DIR = DATA_DIR / "cache"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [121]:
#SQUASHING FROM GITHUB (James)
def main():
    # Start the processing timer.
    process_start = time.perf_counter()

    train_dataset = load_or_build_dataset("train")
    test_dataset = load_or_build_dataset("test")

    print(f"Train Dataset Shape: {train_dataset.shape}\n")
    print(f"Test Dataset Shape: {test_dataset.shape}\n")

    # Stopping hte processing timer.
    process_end = time.perf_counter()
    elapsed_time = process_end - process_start

    print(f"Elapsed time: {elapsed_time} seconds\n")

def get_cache_path(split: str) -> Path:
    return CACHE_DIR / f"{split}_master.parquet"

def load_or_build_dataset(split: str, force_rebuild: bool = False) -> pd.DataFrame:
    cache_path = get_cache_path(split)

    if cache_path.exists() and not force_rebuild:
        print(f"Loading {split} from cache: {cache_path}")

        return pd.read_parquet(cache_path)

    print(f"Building {split} from raw CSVs if no cache found or force_rebuild=True...")

    df = build_base_dataset(split)

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    df.to_parquet(cache_path, index=False)
    print(f"Cached {split} to: {cache_path}")

    return df

# Load, validate and merge the base table with the depth 0 tables.
def build_base_dataset(split: str) -> pd.DataFrame:
    print(f"Building the base dataset for {split}...\n")

    # Load, validate base table.
    base = load_base(split)
    validate_depth0_table(base, "base")
    print("Loaded base table.\n")

    # Load, validate, merge base with static0 table.
    static_0 = load_static_0(split)
    validate_depth0_table(static_0, "static_0")
    merged = join_static_0(base, static_0)
    print("Merged static_0 into base table.\n")

    # Load, validate, merge base with static cb 0 table.
    static_cb_0 = load_static_cb_0(split)
    validate_depth0_table(static_cb_0, "static_cb_0")
    merged = join_static_cb_0(merged, static_cb_0)
    print("Merged static_cb_0 into base table.\n")

    # Load, aggregate, merge tax_registry_a_1 table.
    tax_registry_a_1 = load_tax_registry_a_1(split)
    tax_registry_agg = aggregate_tax_registry_a_1(tax_registry_a_1)
    merged = join_tax_registry_a_1(merged, tax_registry_agg)
    print("Merged tax_registry_a_1 into base table.\n")

    # Load, aggregate, merge person_1 table.
    person_1 = load_person_1(split)
    person_agg = aggregate_person_1(person_1)
    merged = join_person_1(merged, person_agg)
    print("Merged person_1 into base table.\n")

    # Load, aggregate, merge applprev_1 table.
    applprev_1 = load_applprev_1(split)
    applprev_agg = aggregate_applprev_1(applprev_1)
    merged = join_applprev_1(merged, applprev_agg)
    print("Merged applprev_1 into base table.\n")

    # Load, aggregate, merge credit_bureau_a_1 table.
    credit_bureau_a_1 = load_credit_bureau_a_1(split)
    bureau_agg = aggregate_credit_bureau_a_1(credit_bureau_a_1)
    merged = join_credit_bureau_a_1(merged, bureau_agg)
    print("Merged credit_bureau_a_1 into base table.\n")

    # Thin-file flag with zero rows in the raw (pre-aggregation) bureau table.
    merged["is_thin_file"] = build_thin_file_flag(base, credit_bureau_a_1)
    # Make sure the thin-file flag column actually exists and has the
    # expected boolean values before moving on.
    print(f"Thin-file flag: {merged['is_thin_file']}.\n")

    # Load, aggregate (two parts with subrecord to contract to case), merge
    # credit_bureau_a_2 table.
    credit_bureau_a_2 = load_credit_bureau_a_2(split)
    contract_agg = aggregate_credit_bureau_a_2_to_contract(credit_bureau_a_2)
    bureau_a2_case_agg = aggregate_credit_bureau_a_2_to_case(contract_agg)
    merged = join_credit_bureau_a_2(merged, bureau_a2_case_agg)
    print("Merged credit_bureau_a2_contract_case_agg into base table.\n")

    print(f"Finished building the base dataset for {split}!\n")

    return merged

# Returns the split directory which should
# be either split = "train" or "test".
#   get_split_dir("train")
def get_split_dir(split: str) -> Path:
    return CSV_DIR / split

# Load the base table, one row per loan, for
# the given split.
#   load_base("test")
def load_base(split: str) -> pd.DataFrame:
    path = get_split_dir(split) / f"{split}_base.csv"

    return pd.read_csv(path)

# Sum a group but return NaN, not 0, if every value in the
# group is NaN. Prevents "no records" from looking identical
# to "records summed to zero".
def sum_min_count(x: pd.Series):
    return x.sum(min_count=1)

# Load and concatenate the static_0 table, which are
# split into two files.
#   load_static_0("train")
def load_static_0(split: str) -> pd.DataFrame:
    split_dir = get_split_dir(split)

    part_0 = pd.read_csv(split_dir / f"{split}_static_0_0.csv")
    part_1 = pd.read_csv(split_dir / f"{split}_static_0_1.csv")

    # The ignore index parameter is needed because without it the
    # concantenated DF will keep each part's original row index,
    # meaning possible duplicate index 0 for example. Using it
    # re-numbers the index starting from 0 onward.
    return pd.concat([part_0, part_1], ignore_index=True)

# Validate the depth 0 tables to make sure there are no duplicate IDs.
#   validate_depth0_table(train_base, "base")
def validate_depth0_table(df: pd.DataFrame, name: str) -> None:
    dupes = df["case_id"].duplicated().sum()

    if dupes > 0:
        raise ValueError(f"{name}: found {dupes} duplicate case_id values")

    print(f"{name}: {len(df)} rows, no duplicate case_id, all is good!")

# Join the static 0 table with the base table on case ID.
# This is present only for about 70% of applicants.
#   join_static_0(test_base, test_static_0)
def join_static_0(base: pd.DataFrame, static_0: pd.DataFrame) -> pd.DataFrame:
    # Use Left Outer Join in order to use all keys from the left dataframe
    # and missing matches from the right dataframe are filled with NaN.
    # The one to one validation checks that the join keys are unique on
    # both sides and notifies immediately if, for example, a duplicate
    # case_id snuck in.
    merged = base.merge(static_0, on="case_id", how="left", validate="one_to_one")

    # This is a row count check after merging just in case
    # duplicate keys slipped through the cracks.
    if len(merged) != len(base):
        raise ValueError(
            f"Row count changed after join: base had {len(base)}, "
            f"merged has {len(merged)}"
        )

    return merged

# Join the static cb 0 table with the base table on case ID.
# About 30% of the rows will get NaN because these are the thin-file
# people with no history.
#   join_static_cb_0(train_static_cb_0, train_static_cb_0)
def join_static_cb_0(base: pd.DataFrame, static_cb_0: pd.DataFrame) -> pd.DataFrame:
    # Use Left Outer Join in order to use all keys from the left dataframe
    # and missing matches from the right dataframe are filled with NaN.
    merged = base.merge(static_cb_0, on="case_id", how="left", validate="one_to_one")

    # This is a row count check after merging just in case
    # duplicate keys slipped through the cracks.
    if len(merged) != len(base):
        raise ValueError(
            f"Row count changed after join: base had {len(base)}, "
            f"merged has {len(merged)}"
        )

    return merged

# Load the static cb 0 table.
#   load_static_cb_0("train")
def load_static_cb_0(split: str) -> pd.DataFrame:
    path = get_split_dir(split) / f"{split}_static_cb_0.csv"

    return pd.read_csv(path)

# Load the tax_registry_a_1 table.
# Depth 1, meaning many rows per case_id.
#   load_tax_registry_a_1("train")
def load_tax_registry_a_1(split: str) -> pd.DataFrame:
    path = get_split_dir(split) / f"{split}_tax_registry_a_1.csv"

    return pd.read_csv(path)

# Aggregate tax_registry_a_1 to one row per case_id.
#   aggregate_tax_registry_a_1(train_tax_registry_a_1)
def aggregate_tax_registry_a_1(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby("case_id")
        .agg(
            tax_record_count=("num_group1", "count"),
            tax_amount_sum=("amount_4527230A", sum_min_count),
            tax_amount_mean=("amount_4527230A", "mean"),
            tax_amount_max=("amount_4527230A", "max"),
        )
        .reset_index()
    )

# Join the aggregated tax_registry_a_1 onto the merged base table.
#   join_tax_registry_a_1(merged, tax_registry_agg)
def join_tax_registry_a_1(base: pd.DataFrame, tax_registry_agg: pd.DataFrame) -> pd.DataFrame:
    merged = base.merge(tax_registry_agg, on="case_id", how="left", validate="one_to_one")

    if len(merged) != len(base):
        raise ValueError(
            f"Row count changed after join: base had {len(base)}, "
            f"merged has {len(merged)}"
        )

    return merged

# Load the person_1 table.
# Depth 1, meaning 1 to 2 rows per case_id with applicant + optional co-applicant.
#   load_person_1("train")
def load_person_1(split: str) -> pd.DataFrame:
    path = get_split_dir(split) / f"{split}_person_1.csv"

    return pd.read_csv(path)

# Aggregate person_1 to one row per case_id.
#   aggregate_person_1(train_person_1)
def aggregate_person_1(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby("case_id")
        .agg(
            person_count=("num_group1", "count"),
            person_income_sum=("mainoccupationinc_384A", sum_min_count),
            person_income_mean=("mainoccupationinc_384A", "mean"),
            person_income_max=("mainoccupationinc_384A", "max"),
            employment_mean=("empl_employedtotal_800L", "mean"),
            employment_max=("empl_employedtotal_800L", "max")
        )
        .reset_index()
    )

# Join the aggregated person_1 onto the merged base table.
#   join_person_1(merged, person_agg)
def join_person_1(base: pd.DataFrame, person_agg: pd.DataFrame) -> pd.DataFrame:
    merged = base.merge(person_agg, on="case_id", how="left", validate="one_to_one")

    if len(merged) != len(base):
        raise ValueError(
            f"Row count changed after join: base had {len(base)}, "
            f"merged has {len(merged)}"
        )

    return merged

# Load the applprev_1 table.
# Depth 1, meaning 0 to4 rows per case_id for the applicant's
# own previous applications to this lender.
#   load_applprev_1("train")
def load_applprev_1(split: str) -> pd.DataFrame:
    path = get_split_dir(split) / f"{split}_applprev_1.csv"

    return pd.read_csv(path)

# Aggregate applprev_1 to one row per case_id.
#   aggregate_applprev_1(train_applprev_1)
def aggregate_applprev_1(df: pd.DataFrame) -> pd.DataFrame:
    # Make a copy so as to not mutate the original DataFrame.
    df = df.copy()
    # Conver the string dates to real datetimes so our min/max will
    # sort chronologically and not alphabetically.
    # Any value that cannot be parsed will become NaT, Not a Time,
    # instead of crashing.
    df["approvaldate_319D"] = pd.to_datetime(df["approvaldate_319D"], errors="coerce")

    return (
        df.groupby("case_id")
        .agg(
            prev_application_count=("num_group1", "count"),
            prev_credit_amount_sum=("credamount_590A", sum_min_count),
            prev_credit_amount_mean=("credamount_590A", "mean"),
            prev_credit_amount_max=("credamount_590A", "max"),
            prev_approval_date_min=("approvaldate_319D", "min"),
            prev_approval_date_max=("approvaldate_319D", "max"),
        )
        .reset_index()
    )

# Join the aggregated applprev_1 onto the merged base table.
#   join_applprev_1(merged, applprev_agg)
def join_applprev_1(base: pd.DataFrame, applprev_agg: pd.DataFrame) -> pd.DataFrame:
    merged = base.merge(applprev_agg, on="case_id", how="left", validate="one_to_one")

    if len(merged) != len(base):
        raise ValueError(
            f"Row count changed after join: base had {len(base)}, "
            f"merged has {len(merged)}"
        )

    return merged

# Load and concatenate the credit_bureau_a_1 table, split into two files.
# This is the most important signal table where the thin-file applicants
# have zero rows in.
#   load_credit_bureau_a_1("train")
def load_credit_bureau_a_1(split: str) -> pd.DataFrame:
    split_dir = get_split_dir(split)

    part_0 = pd.read_csv(split_dir / f"{split}_credit_bureau_a_1_0.csv")
    part_1 = pd.read_csv(split_dir / f"{split}_credit_bureau_a_1_1.csv")

    return pd.concat([part_0, part_1], ignore_index=True)

# Aggregate credit_bureau_a_1 to one row per case_id.
#   aggregate_credit_bureau_a_1(train_credit_bureau_a_1)
def aggregate_credit_bureau_a_1(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby("case_id")
        .agg(
            bureau_contract_count=("num_group1", "count"),
            bureau_credit_amount_sum=("credamount_770A", sum_min_count),
            bureau_credit_amount_mean=("credamount_770A", "mean"),
            bureau_credit_amount_max=("credamount_770A", "max"),
            bureau_overdue_amount_mean=("overdueamountmax_950A", "mean"),
            bureau_overdue_amount_max=("overdueamountmax_950A", "max"),
            bureau_dpd_max=("pmts_dpdvalue_108P", "max"),
            bureau_dpd_mean=("pmts_dpdvalue_108P", "mean"),
        )
        .reset_index()
    )

# Helper to compute the thin-file flag.
# True if this case_id has zero rows in the raw pre-aggregation credit_bureau_a_1 table,
# meaning no bureau history at all.
#   build_thin_file_flag(train_base, train_credit_bureau_a_1)
def build_thin_file_flag(base: pd.DataFrame, credit_bureau_a_1: pd.DataFrame) -> pd.Series:
    has_bureau = base["case_id"].isin(credit_bureau_a_1["case_id"])

    is_thin_file = has_bureau == False

    return is_thin_file

# Join the aggregated credit_bureau_a_1 onto the merged base table.
#   join_credit_bureau_a_1(merged, bureau_agg)
def join_credit_bureau_a_1(base: pd.DataFrame, bureau_agg: pd.DataFrame) -> pd.DataFrame:
    merged = base.merge(bureau_agg, on="case_id", how="left", validate="one_to_one")

    if len(merged) != len(base):
        raise ValueError(
            f"Row count changed after join: base had {len(base)}, "
            f"merged has {len(merged)}"
        )

    return merged

# Load the credit_bureau_a_2 table.
# Depth 2, meaning case_id to num_group1 (contract) to
# num_group2 (payment/subrecord within that contract) which is the largest table.
#   load_credit_bureau_a_2("train")
def load_credit_bureau_a_2(split: str) -> pd.DataFrame:
    path = get_split_dir(split) / f"{split}_credit_bureau_a_2.csv"

    return pd.read_csv(path)

# Part 1: Collapse credit_bureau_a_2 from subrecord level (case_id, num_group1, num_group2) to
# contract level (case_id, num_group1). One row per past credit
# contract by summarizing its payment history.
#   aggregate_credit_bureau_a_2_to_contract(train_credit_bureau_a_2)
def aggregate_credit_bureau_a_2_to_contract(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby(["case_id", "num_group1"])
        .agg(
            contract_subrecord_count=("num_group2", "count"),
            contract_overdue_sum=("pmts_overdue_1140A", sum_min_count),
            contract_overdue_max=("pmts_overdue_1140A", "max"),
            contract_overdue_valid_count=("pmts_overdue_1140A", "count"),
        )
        .reset_index()
    )

# Part 2: Collapse contract-level credit_bureau_a_2 summaries from (case_id, num_group1)
# to one row per case_id.
#   aggregate_credit_bureau_a_2_to_case(contract_agg)
def aggregate_credit_bureau_a_2_to_case(contract_agg: pd.DataFrame) -> pd.DataFrame:
    case_agg = (
        contract_agg.groupby("case_id")
        .agg(
            bureau_a2_contract_count=("num_group1", "count"),
            bureau_a2_record_count=("contract_subrecord_count", "sum"),
            bureau_a2_overdue_sum=("contract_overdue_sum", sum_min_count),
            bureau_a2_overdue_max=("contract_overdue_max", "max"),
            bureau_a2_overdue_valid_count=("contract_overdue_valid_count", "sum"),
        )
        .reset_index()
    )

    # Weighted mean, computed after. aggregation. This is the total overdue divided by
    # total valid records and not the average of each contract's own mean.
    # This avoids overweighting contracts that happen to have fewer subrecords.
    case_agg["bureau_a2_overdue_mean"] = (
        case_agg["bureau_a2_overdue_sum"] / case_agg["bureau_a2_overdue_valid_count"]
    )

    return case_agg

# Join both aggregated credit_bureau_a_2 parts into the merged base table.
#   join_credit_bureau_a_2(merged, bureau_a2_case_agg)
def join_credit_bureau_a_2(base: pd.DataFrame, bureau_a2_case_agg: pd.DataFrame) -> pd.DataFrame:
    merged = base.merge(bureau_a2_case_agg, on="case_id", how="left", validate="one_to_one")

    if len(merged) != len(base):
        raise ValueError(
            f"Row count changed after join: base had {len(base)}, "
            f"merged has {len(merged)}"
        )

    return merged

if __name__ == "__main__":
    main()

Loading train from cache: /content/drive/MyDrive/AMEX1B - Data /cache/train_master.parquet
Loading test from cache: /content/drive/MyDrive/AMEX1B - Data /cache/test_master.parquet
Train Dataset Shape: (1000000, 46)

Test Dataset Shape: (200000, 45)

Elapsed time: 2.0971706560012535 seconds



In [122]:
df_train_master = pd.read_parquet(CACHE_DIR / "train_master.parquet")
df_train_master.shape

(1000000, 46)

In [123]:
df_train_master.columns.tolist()

['case_id',
 'date_decision',
 'MONTH',
 'WEEK_NUM',
 'target',
 'mainoccupationinc_384A',
 'credamount_770A',
 'annuity_780A',
 'days_employed_700P',
 'education_927M',
 'maritalstatus_703M',
 'birth_259D',
 'riskassesment_940T',
 'numberofqueries_146L',
 'description_5085714M',
 'tax_record_count',
 'tax_amount_sum',
 'tax_amount_mean',
 'tax_amount_max',
 'person_count',
 'person_income_sum',
 'person_income_mean',
 'person_income_max',
 'employment_mean',
 'employment_max',
 'prev_application_count',
 'prev_credit_amount_sum',
 'prev_credit_amount_mean',
 'prev_credit_amount_max',
 'prev_approval_date_min',
 'prev_approval_date_max',
 'bureau_contract_count',
 'bureau_credit_amount_sum',
 'bureau_credit_amount_mean',
 'bureau_credit_amount_max',
 'bureau_overdue_amount_mean',
 'bureau_overdue_amount_max',
 'bureau_dpd_max',
 'bureau_dpd_mean',
 'is_thin_file',
 'bureau_a2_contract_count',
 'bureau_a2_record_count',
 'bureau_a2_overdue_sum',
 'bureau_a2_overdue_max',
 'bureau_a2_ove

In [124]:
#View spread of data and quality
import pandas as pd

print("MISSING VALUES")
# calculate count and percentage of missing values
missing_df = pd.DataFrame({
    'Missing Count': df_train_master.isna().sum(),
    'Missing %': (df_train_master.isna().mean() * 100).round(2)
})
display(missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing %', ascending=False))

print("DATA TYPES")
# show data types to spot incorrect ones (like numeric data stored as objects)
display(df_train_master.dtypes.value_counts())
display(df_train_master.dtypes)

print("SPREAD OF DATA")
# generate min, max, mean, and quartiles to understand spread
display(df_train_master.describe().T)

print("INCORRECT VALUES")
# check for invalid data
invalid_checks = {
    "Negative Age": (df_train_master.get('age', pd.Series(dtype=float)) < 0).sum(),
    "Age > 120 (Unrealistic)": (df_train_master.get('age', pd.Series(dtype=float)) > 120).sum(),
    "Negative Income (Main)": (df_train_master.get('mainoccupationinc_384A', pd.Series(dtype=float)) < 0).sum(),
    "Negative Credit Amount": (df_train_master.get('credamount_770A', pd.Series(dtype=float)) < 0).sum(),
    "Negative Annuity": (df_train_master.get('annuity_780A', pd.Series(dtype=float)) < 0).sum(),
    "Negative Days Employed": (df_train_master.get('days_employed_700P', pd.Series(dtype=float)) < 0).sum()
}

for check_name, count in invalid_checks.items():
    if count > 0:
        print(f"{check_name}: {count} rows found")
    else:
        print(f"{check_name}: 0 rows found (Clean)")

MISSING VALUES


,Missing Count,Missing %
bureau_a2_overdue_max,359745,35.97
bureau_a2_overdue_mean,359745,35.97
bureau_a2_overdue_sum,359745,35.97
bureau_overdue_amount_mean,337107,33.71
bureau_overdue_amount_max,337107,33.71
riskassesment_940T,334965,33.50
bureau_a2_record_count,329111,32.91
bureau_a2_overdue_valid_count,329111,32.91
bureau_a2_contract_count,329111,32.91
numberofqueries_146L,299821,29.98


DATA TYPES


,count
float64,33
int64,5
object,5
datetime64[ns],2
bool,1


,0
case_id,int64
date_decision,object
MONTH,int64
WEEK_NUM,int64
target,int64
mainoccupationinc_384A,float64
credamount_770A,float64
annuity_780A,float64
days_employed_700P,float64
education_927M,object


SPREAD OF DATA


,count,mean,min,25%,50%,75%,max,std
case_id,1000000.0,500000.5,1.0,250000.75,500000.5,750000.25,1000000.0,288675.278933
MONTH,1000000.0,201948.289559,201901.0,201906.0,201911.0,202004.0,202009.0,48.800468
WEEK_NUM,1000000.0,45.498973,0.0,23.0,45.0,68.0,91.0,26.560375
target,1000000.0,0.191752,0.0,0.0,0.0,0.0,1.0,0.393679
mainoccupationinc_384A,950349.0,24477.415393,2515.0,16139.0,22021.0,30008.0,174473.0,11911.971994
credamount_770A,980041.0,16730.809971,1447.0,10527.0,14760.0,20693.0,150395.0,8932.87458
annuity_780A,900323.0,1242.750132,79.0,782.0,1097.0,1538.0,12069.0,661.880406
days_employed_700P,849429.0,1524.346248,0.0,861.0,1502.0,2141.0,5852.0,902.202133
riskassesment_940T,665035.0,0.49977,0.0102,0.3373,0.4996,0.6624,0.9939,0.208339
numberofqueries_146L,700179.0,2.997915,0.0,2.0,3.0,4.0,18.0,1.998898


INCORRECT VALUES
Negative Age: 0 rows found (Clean)
Age > 120 (Unrealistic): 0 rows found (Clean)
Negative Income (Main): 0 rows found (Clean)
Negative Credit Amount: 0 rows found (Clean)
Negative Annuity: 0 rows found (Clean)
Negative Days Employed: 0 rows found (Clean)


In [125]:
# CLEANING & FORMATTING

# Converting date columns from object to date time format
date_cols = ["date_decision", "birth_259D", "prev_approval_date_min", "prev_approval_date_max"]
for col in date_cols:
  if col in df_train_master.columns:
    df_train_master[col] = pd.to_datetime(df_train_master[col], errors='coerce')

# Printing out updated date type counts
print("Data types: ")
display(df_train_master.dtypes.value_counts())
display(df_train_master.dtypes)

# formatting categorical strings and rows that contain text labels
categorical_cols = df_train_master.select_dtypes(include=['object', 'category']).columns.tolist()
for col in categorical_cols:
  valid_text_labels = df_train_master[col].notna()

  # lowercase, trimming spaces, and changing spaces between words to hyphens (' ') -> ('-')
  df_train_master.loc[valid_text_labels, col] = (df_train_master.loc[valid_text_labels, col].astype(str).str.lower().str.strip().str.replace(r'\s+', '-', regex=True))

  # replacing text words in unknown values list with nan, keeping NA's for now to train the model and see how it works and then deciding later based on results
  unknown_values = ['nan', 'none', 'null', '', 'unknown', 'missing', 'missing_value']
  df_train_master[col] = df_train_master[col].replace(unknown_values, np.nan)
print(f"Formatted and Clean categorical columns: {len(categorical_cols)}")

# printing out missing values
display(df_train_master[categorical_cols].isna().sum().to_frame(name = "missing values count"))

Data types: 


,count
float64,33
int64,5
datetime64[ns],4
object,3
bool,1


,0
case_id,int64
date_decision,datetime64[ns]
MONTH,int64
WEEK_NUM,int64
target,int64
mainoccupationinc_384A,float64
credamount_770A,float64
annuity_780A,float64
days_employed_700P,float64
education_927M,object


Formatted and Clean categorical columns: 3


,missing values count
education_927M,0
maritalstatus_703M,0
description_5085714M,299821


In [126]:
# Feature Engineering STEP 1: IDENTIFY FEATURE TYPES AND TARGET

# Identify ID and target columns
id_col = "case_id"
target_col = "target"

print("Number of rows:", df_train_master.shape[0])
print("Number of columns:", df_train_master.shape[1])

print("\nID column:", id_col)
print("Target column:", target_col)
print("0 = repaid")
print("1 = default")

Number of rows: 1000000
Number of columns: 46

ID column: case_id
Target column: target
0 = repaid
1 = default


In [127]:
# Identify feature types
numeric_cols = df_train_master.select_dtypes(include=["number"]).columns.tolist()

categorical_cols = df_train_master.select_dtypes(include=["object", "category"]).columns.tolist()

date_cols = df_train_master.select_dtypes(include=["datetime64[ns]"]).columns.tolist()

boolean_cols = df_train_master.select_dtypes(include=["bool"]).columns.tolist()

# case_id and target are numeric columns,
# but they are not numerical predictor features
numeric_features = [
    col for col in numeric_cols
    if col not in [id_col, target_col]
]

print("\nNumeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_cols))
print(categorical_cols)

print("\nDate features:", len(date_cols))
print(date_cols)

print("\nBoolean features:", len(boolean_cols))
print(boolean_cols)


Numeric features: 36
['MONTH', 'WEEK_NUM', 'mainoccupationinc_384A', 'credamount_770A', 'annuity_780A', 'days_employed_700P', 'riskassesment_940T', 'numberofqueries_146L', 'tax_record_count', 'tax_amount_sum', 'tax_amount_mean', 'tax_amount_max', 'person_count', 'person_income_sum', 'person_income_mean', 'person_income_max', 'employment_mean', 'employment_max', 'prev_application_count', 'prev_credit_amount_sum', 'prev_credit_amount_mean', 'prev_credit_amount_max', 'bureau_contract_count', 'bureau_credit_amount_sum', 'bureau_credit_amount_mean', 'bureau_credit_amount_max', 'bureau_overdue_amount_mean', 'bureau_overdue_amount_max', 'bureau_dpd_max', 'bureau_dpd_mean', 'bureau_a2_contract_count', 'bureau_a2_record_count', 'bureau_a2_overdue_sum', 'bureau_a2_overdue_max', 'bureau_a2_overdue_valid_count', 'bureau_a2_overdue_mean']

Categorical features: 3
['education_927M', 'maritalstatus_703M', 'description_5085714M']

Date features: 4
['date_decision', 'birth_259D', 'prev_approval_date_m

In [128]:
# Examine target distribution
target_counts = df_train_master[target_col].value_counts().sort_index()
target_percentages = ( df_train_master[target_col].value_counts(normalize=True).sort_index().mul(100))

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": target_percentages
})

print("\nTARGET DISTRIBUTION")
display(target_summary)


TARGET DISTRIBUTION


,Count,Percentage
target,,
0,808248,80.8248
1,191752,19.1752


The target variable is imbalanced. Roughly 81% repaid vs. 19% default.

In [129]:
# Step 2: Seperate X and y
# X contains all the columns except for target
# y only contains the target column

target_col = "target"
id_col = "case_id"

y = df_train_master[target_col]
X = df_train_master.drop(columns=[target_col])

In [130]:
#Step 3: Split into train/validation using WEEK_NUM

#Look at the column
print(X["WEEK_NUM"].min(), X["WEEK_NUM"].max())
print(X["WEEK_NUM"].value_counts().sort_index())

0 91
WEEK_NUM
0     10769
1     10722
2     10804
3     10980
4     11049
      ...  
87    10973
88    10844
89    11037
90    10880
91    10940
Name: count, Length: 92, dtype: int64


Train set runs weeks 0–91

We chose Week 80 as the cutoff because it allows us to use Weeks 0–79 for training while saving the last 12 weeks, Weeks 80–91, for validation. This gives the model a large amount of earlier data to learn from while testing it on a separate, later time period.

We chose to split the data at this stage to reduce the risk of data leakage during feature engineering and preprocessing. By separating the training and validation data early, we can create the same features for both sets while making sure that any decisions learned from the data, such as imputation values, winsorization thresholds, or encoding, are based only on the training set.

In [131]:
#Split into train and validation
split_week = 80

# Earlier weeks = training
train_rows = X["WEEK_NUM"] < split_week

#Later weeks = validation
val_rows = X["WEEK_NUM"] >= split_week

X_train = X.loc[train_rows].copy()
X_val = X.loc[val_rows].copy()

y_train = y.loc[train_rows].copy()
y_val = y.loc[val_rows].copy()

In [132]:
#Check the split

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nX_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

print("\nTraining weeks:",
      X_train["WEEK_NUM"].min(), "to", X_train["WEEK_NUM"].max())

print("Validation weeks:",
      X_val["WEEK_NUM"].min(), "to", X_val["WEEK_NUM"].max())

X_train shape: (869559, 45)
y_train shape: (869559,)

X_val shape: (130441, 45)
y_val shape: (130441,)

Training weeks: 0 to 79
Validation weeks: 80 to 91


In [133]:
def add_applicant_feat(df):
  df = df.copy()

  # Make sure the date columns are actual dates
  df["date_decision"] = pd.to_datetime(df["date_decision"], errors="coerce")
  df["birth_259D"] = pd.to_datetime(df["birth_259D"], errors="coerce")
  df["prev_approval_date_min"] = pd.to_datetime(df["prev_approval_date_min"], errors="coerce")
  df["prev_approval_date_max"] = pd.to_datetime(df["prev_approval_date_max"], errors="coerce")

  # ============================================================
  # FAMILY 7: Demographic / derived
  # ============================================================

  # Age at the time of the decision
  df["age_years"] = (df["date_decision"] - df["birth_259D"]).dt.days / 365.25

  # Employment history relative to age
  employment_years = df["days_employed_700P"] / 365.25

  df["employment_age_ratio"] = (employment_years / df["age_years"].replace(0, np.nan))

  # Income relative to employment length
  df["income_per_employed_year"] = ( df["mainoccupationinc_384A"]/ (df["days_employed_700P"] / 365.0 + 1))

  # Relative income contribution
  df["coapplicant_income_ratio"] = np.where(
    (df["person_count"] > 1)
    & (df["person_income_min"] > 0)
    & (df["person_income_max"] > 0),
    df["person_income_min"] / df["person_income_max"].replace(0, np.nan),
    np.nan
  )

  # ============================================================
  # FAMILY 5: AFFORDABILITY AND LEVERAGE
  # ============================================================

  # Prevent division by 0
  income = df["mainoccupationinc_384A"].replace(0, np.nan)
  credit = df["credamount_770A"].replace(0, np.nan)
  annuity = df["annuity_780A"].replace(0, np.nan)
  bureau_credit = df["bureau_credit_amount_mean"].replace(0, np.nan)

  # Affordability ratios
  df["credit_income_ratio"] = df["credamount_770A"] / income
  df["annuity_income_ratio"] = df["annuity_780A"] / income
  df["implied_term"] = df["credamount_770A"] / annuity

  # Compare average tax-record amount to declared income
  df["tax_declared_income_ratio"] = df["tax_amount_mean"] / income

  # Total bureau credit relative to income
  df["bureau_exposure_ratio"] = (df["bureau_credit_amount_sum"] / income)

  #Largest bureau credit compared with total
  df["credit_concentration"] = (df["bureau_credit_amount_max"] / df["bureau_credit_amount_sum"].replace(0, np.nan))

  df["prior_credit_to_current"] = (df["prev_credit_amount_mean"] / credit)

  # ============================================================
  # FAMILY 4: COVERAGE AND PRESENCE
  # ============================================================

  # Household
  df["has_coapplicant"] = (df["person_count"] > 1).astype(int)

  # Has previous application history
  df["has_prev"] = (df["prev_application_count"].fillna(0) > 0).astype(int)

  # Has tax records
  df["has_tax"] = (df["tax_record_count"].fillna(0) > 0).astype(int)

  # Number of historical data sources available
  df["history_source_count"] = (
    (~df["is_thin_file"].astype(bool)).astype(int)
    + df["has_prev"]
    + df["has_tax"]
  )

  # ============================================================
  # FAMILY 6: CREDIT BEHAVIOR
  # ============================================================

  # Bureau severity relative to credit
  df["bureau_overdue_credit_ratio"] = (df["bureau_overdue_amount_mean"] / bureau_credit)

  # Bureau queries relative to contracts
  df["queries_per_contract"] = (df["numberofqueries_146L"] / (df["bureau_contract_count"].fillna(0) + 1))

  # Worst overdue amount relative to income
  df["max_overdue_to_income"] = (df["bureau_overdue_amount_max"] / income)

  #Payment records per bureau contract
  df["months_per_contract"] = (df["bureau_a2_record_count"]/ df["bureau_contract_count"].replace(0, np.nan))

  #Average overdue amount per contract
  df["avg_overdue_per_contract"] = ( df["bureau_overdue_amount_sum"] / df["bureau_contract_count"].replace(0, np.nan) )

  # Previous application recency
  df["prev_days_since_last_approval"] = (df["date_decision"] - df["prev_approval_date_max"]).dt.days
  df["prev_days_since_first_approval"] = (df["date_decision"] - df["prev_approval_date_min"]).dt.days


  # Compare current requested credit to combined credit history
  combined_credit_history = (df["bureau_credit_amount_sum"].fillna(0) + df["prev_credit_amount_sum"].fillna(0))

  df["current_history_credit_ratio"] = (df["credamount_770A"] / combined_credit_history.replace(0, np.nan))

  # Split version of the combined ratio, so the model can see which source (bureau vs. prior applications) is actually driving the number
  df["credit_vs_bureau_history"] = df["credamount_770A"] / bureau_credit

  return df


def payment_overdue_share(split):
  cb2_raw = load_credit_bureau_a_2(split).copy()

  overdue = cb2_raw["pmts_overdue_1140A"]

  # 1 = overdue, 0 = not overdue, NaN = unknown
  cb2_raw["is_overdue"] = np.where(
    overdue.notna(),
    (overdue > 0).astype(float),
    np.nan
  )

  result = (
    cb2_raw.groupby("case_id")["is_overdue"]
    .mean()
    .reset_index()
  )

  return result.rename(columns={"is_overdue": "pay_overdue_share"})

# ============================================================
# FAMILY 3: NEW DEPTH-1 AGGREGATES
# ============================================================

def get_additional_aggregates(split):

  tax_raw = load_tax_registry_a_1(split)
  prev_raw = load_applprev_1(split)
  person_raw = load_person_1(split)
  cb1_raw = load_credit_bureau_a_1(split)

  tax_feats = (
      tax_raw.groupby("case_id")["amount_4527230A"].agg(["std", "min"]).reset_index()
      .rename(columns={
          "std": "tax_amount_std",
          "min": "tax_amount_min"
      })
  )
  prev_feats = (
      prev_raw.groupby("case_id")["credamount_590A"].agg(["std", "min"]).reset_index()
      .rename(columns={
          "std": "prev_credit_amount_std",
          "min": "prev_credit_amount_min"
      })
  )

  person_feats = (
      person_raw.groupby("case_id")["mainoccupationinc_384A"].min().reset_index()
      .rename(columns={
          "mainoccupationinc_384A": "person_income_min"
      })
  )

  cb1_raw["overdueamountmax_950A"] = pd.to_numeric(cb1_raw["overdueamountmax_950A"], errors="coerce")

  bureau_feats = (
    cb1_raw.groupby("case_id")["overdueamountmax_950A"]
    .sum(min_count=1)
    .reset_index()
    .rename(columns={
        "overdueamountmax_950A": "bureau_overdue_amount_sum"
    })
  )

  return tax_feats, prev_feats, person_feats, bureau_feats

def get_contract_behavior_features(split):
  cb1_raw = load_credit_bureau_a_1(split).copy()

  dpd = pd.to_numeric(cb1_raw["pmts_dpdvalue_108P"], errors="coerce")
  overdue = pd.to_numeric(cb1_raw["overdueamountmax_950A"], errors="coerce")

  cb1_raw["contract_late"] = np.where(dpd.notna(), (dpd > 0).astype(float), np.nan)

  cb1_raw["contract_overdue"] = np.where(overdue.notna(), (overdue > 0).astype(float), np.nan)

  result = (cb1_raw.groupby("case_id").agg(
      dpd_contract_rate=("contract_late", "mean"),
      overdue_contract_rate=("contract_overdue", "mean")
      ).reset_index())

  return result

# ============================================================
# FAMILY 8: Bureau contract classification
# ============================================================

def get_contract_classification(split):
  cb1_raw = load_credit_bureau_a_1(split)

  result = (
      cb1_raw.dropna(subset=["classificationofcontr_13M"]).groupby("case_id")["classificationofcontr_13M"]
      .agg(lambda x: x.mode().iloc[0])
      .reset_index()
  )

  return result

# ============================================================
# FAMILY 9: Missingness indicators
# ============================================================
def add_missingness_feat(df):
  df = df.copy()

  df["missing_prev_history"] = df["prev_application_count"].isna().astype(int)
  df["missing_tax_history"] = df["tax_record_count"].isna().astype(int)
  df["missing_income"] = df["mainoccupationinc_384A"].isna().astype(int)
  df["missing_annuity"] = df["annuity_780A"].isna().astype(int)
  df["missing_days_employed"] = df["days_employed_700P"].isna().astype(int)
  df["missing_overdue"] = df["bureau_overdue_amount_mean"].isna().astype(int)

  return df

In [135]:
# Apply it to both sets
payment_overdue_feats = payment_overdue_share("train")

tax_extra_feats, prev_extra_feats, person_extra_feats, bureau_extra_feats = get_additional_aggregates("train")

contract_behavior_feats = get_contract_behavior_features("train")

contract_class_feats = get_contract_classification("train")


# Raw-table feature sets to merge
feature_tables = [
  payment_overdue_feats,
  tax_extra_feats,
  prev_extra_feats,
  person_extra_feats,
  bureau_extra_feats,
  contract_behavior_feats,
  contract_class_feats
]


# Save original applicant order before merging
train_case_ids = X_train["case_id"].copy().reset_index(drop=True)
val_case_ids = X_val["case_id"].copy().reset_index(drop=True)

# Merge into X_train
for feats in feature_tables:
  new_cols = [c for c in feats.columns
              if c != "case_id" and c not in X_train.columns
  ]

  if new_cols:
    X_train = X_train.merge(
      feats[["case_id"] + new_cols],
      on="case_id",
      how="left",
      validate="many_to_one"
  )


# Merge into X_val
for feats in feature_tables:
  new_cols = [c for c in feats.columns
              if c != "case_id" and c not in X_val.columns
  ]

  if new_cols:
    X_val = X_val.merge(
      feats[["case_id"] + new_cols],
      on="case_id",
      how="left",
      validate="many_to_one"
    )

X_train = add_applicant_feat(X_train)
X_val = add_applicant_feat(X_val)

X_train = add_missingness_feat(X_train)
X_val = add_missingness_feat(X_val)

# Check that merging did not change applicant order
print("Train order unchanged:",
      X_train["case_id"].reset_index(drop=True).equals(train_case_ids))

print("Val order unchanged:",
      X_val["case_id"].reset_index(drop=True).equals(val_case_ids))


print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Same columns:", X_train.columns.equals(X_val.columns))

Train order unchanged: True
Val order unchanged: True
X_train: (869559, 85)
X_val: (130441, 85)
Same columns: True


In [136]:
#Check the new columns

# Features created in add_applicant_feat() + raw-table features

new_feature_cols = [
    # Family 7: Demographic / derived
    "age_years",
    "employment_age_ratio",
    "income_per_employed_year",
    "coapplicant_income_ratio",

    # Family 5: Affordability and leverage
    "credit_income_ratio",
    "annuity_income_ratio",
    "implied_term",
    "tax_declared_income_ratio",
    "bureau_exposure_ratio",
    "credit_concentration",
    "prior_credit_to_current",

    # Family 4: Coverage and presence
    "has_coapplicant",
    "has_prev",
    "has_tax",
    "history_source_count",

    # Family 6: Credit behavior
    "bureau_overdue_credit_ratio",
    "queries_per_contract",
    "max_overdue_to_income",
    "months_per_contract",
    "avg_overdue_per_contract",
    "dpd_contract_rate",
    "overdue_contract_rate",
    "pay_overdue_share",

    # Previous application history
    "prev_days_since_last_approval",
    "prev_days_since_first_approval",
    "current_history_credit_ratio",
    "credit_vs_bureau_history",

    # Family 3: Depth-1 aggregates
    "tax_amount_std",
    "tax_amount_min",
    "prev_credit_amount_std",
    "prev_credit_amount_min",
    "person_income_min",
    "bureau_overdue_amount_sum",

    # Family 8: Categorical features
    "education_927M",
    "description_5085714M",
    "classificationofcontr_13M",
]

# Categorical columns
categorical_cols = [
    "education_927M",
    "description_5085714M",
    "classificationofcontr_13M"
]

# Numerical columns
numerical_cols = [
    c for c in new_feature_cols
    if c not in categorical_cols
    and c in X_train.columns
]

# Numerical feature summary
display(X_train[numerical_cols].describe())

# Check for features not yet added
missing_features = [
    c for c in new_feature_cols
    if c not in X_train.columns
]

print("Missing features:", missing_features)

,age_years,employment_age_ratio,income_per_employed_year,coapplicant_income_ratio,credit_income_ratio,annuity_income_ratio,implied_term,tax_declared_income_ratio,bureau_exposure_ratio,credit_concentration,...,prev_days_since_last_approval,prev_days_since_first_approval,current_history_credit_ratio,credit_vs_bureau_history,tax_amount_std,tax_amount_min,prev_credit_amount_std,prev_credit_amount_min,person_income_min,bureau_overdue_amount_sum
count,869559.000000,738636.000000,701965.000000,434186.000000,809827.000000,743965.000000,767326.000000,619608.000000,578521.000000,608739.000000,...,676024.000000,676024.000000,801140.000000,596508.000000,434806.000000,651988.000000,521038.000000,695534.000000,846780.000000,576371.000000
mean,46.190479,0.100554,6673.636199,0.676136,0.844530,0.062784,17.291945,1.198786,3.584498,0.450376,...,969.191714,1612.173520,0.507083,1.234300,7993.832943,19740.720599,4988.613797,6616.413816,21657.776598,962.764440
std,14.138730,0.072089,6569.947116,0.225622,0.647079,0.048218,13.966815,0.713795,3.135746,0.249043,...,534.335055,534.102846,0.892601,0.865360,6105.211402,8416.193406,4094.119740,4442.122053,10460.284694,838.549657
min,20.991102,0.000000,325.597132,0.046265,0.022989,0.002273,0.519103,0.062520,0.024893,0.136074,...,30.000000,30.000000,0.009463,0.036726,0.000000,3262.000000,0.000000,380.000000,2767.000000,0.000000
25%,33.952088,0.049675,2962.761506,0.500104,0.423915,0.031476,8.345817,0.713679,1.413043,0.266721,...,554.000000,1259.750000,0.132453,0.673674,3623.922254,14020.000000,2271.934088,3880.000000,14497.000000,320.000000
50%,46.173854,0.088873,4585.087623,0.685008,0.670375,0.049839,13.461142,1.031836,2.755082,0.361226,...,865.000000,1717.000000,0.243045,1.015667,6680.592663,18041.000000,4002.787399,5502.000000,19419.000000,775.000000
75%,58.444901,0.137305,7628.048369,0.871380,1.058755,0.078829,21.686522,1.491804,4.786885,0.549443,...,1324.000000,2026.000000,0.511226,1.535441,10805.094312,23441.000000,6500.141971,7935.000000,26216.000000,1407.000000
max,71.460643,0.719737,124715.000000,1.000000,20.641617,1.338481,390.714286,15.988704,66.724163,1.000000,...,2552.000000,2552.000000,48.884956,40.811368,89970.296037,147877.000000,107284.362162,174258.000000,193361.000000,8830.000000


Missing features: []


In [137]:
# Examine categorical features (Nitya)
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Number of categorical features:", len(categorical_cols))
print(categorical_cols)

Number of categorical features: 4
['education_927M', 'maritalstatus_703M', 'description_5085714M', 'classificationofcontr_13M']


In [139]:
# Examine category counts and missing values (Nitya)
for col in categorical_cols:
    print("\n" + "=" * 60)
    print("COLUMN:", col)
    print("Unique categories:", X_train[col].nunique(dropna=True))
    print("Missing:", X_train[col].isna().sum())
    print(X_train[col].value_counts(dropna=False).head(15))


COLUMN: education_927M
Unique categories: 5
Missing: 0
education_927M
6def22f0    243012
242b264b    190097
10795bac    189923
2d4a2bc4    123532
f937ecaa    122995
Name: count, dtype: int64

COLUMN: maritalstatus_703M
Unique categories: 4
Missing: 0
maritalstatus_703M
e78cb9c0    218020
b58630a7    217946
f62a4fc7    216803
ea6fb4b5    216790
Name: count, dtype: int64

COLUMN: description_5085714M
Unique categories: 6
Missing: 260820
description_5085714M
None        260820
521190c5    124197
3802c3d2    123852
4dc79565     94124
6ef3de40     94075
5568a5e6     86485
34bbcd2a     86006
Name: count, dtype: int64

COLUMN: classificationofcontr_13M
Unique categories: 6
Missing: 260820
classificationofcontr_13M
NaN         260820
ba38dde4    127410
01851564    118427
d8bf6dd7    114311
6f5666c7    100542
54ddc605     81331
fe64f125     66718
Name: count, dtype: int64


In [140]:
# Check for rare categories (Nitya)
for col in categorical_cols:
    percentages = (
        X_train[col]
        .value_counts(normalize=True, dropna=False) * 100
    )

    rare = percentages[percentages < 1]

    print("\n" + "=" * 60)
    print("COLUMN:", col)
    print("Total categories:", len(percentages))
    print("Categories below 1%:", len(rare))
    print(rare.head(20))


COLUMN: education_927M
Total categories: 5
Categories below 1%: 0
Series([], Name: proportion, dtype: float64)

COLUMN: maritalstatus_703M
Total categories: 4
Categories below 1%: 0
Series([], Name: proportion, dtype: float64)

COLUMN: description_5085714M
Total categories: 7
Categories below 1%: 0
Series([], Name: proportion, dtype: float64)

COLUMN: classificationofcontr_13M
Total categories: 7
Categories below 1%: 0
Series([], Name: proportion, dtype: float64)


In [141]:
# Identify numerical features (Nitya)
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

# Do not use case_id as a model feature
numeric_features = [
    col for col in numeric_cols
    if col != "case_id"
]

print("Number of numerical features:", len(numeric_features))

Number of numerical features: 75


In [142]:
# Examine numerical features (Nitya)
numeric_summary = X_train[numeric_features].describe().T

numeric_summary["missing"] = X_train[numeric_features].isna().sum()
numeric_summary["missing_pct"] = (
    X_train[numeric_features].isna().mean() * 100
)
numeric_summary["skew"] = X_train[numeric_features].skew()

display(numeric_summary)

,count,mean,std,min,25%,50%,75%,max,missing,missing_pct,skew
MONTH,869559.0,201939.307094,46.044312,201901.0,201905.0,201910.0,202002.0,202007.0,0,0.000000,0.677701
WEEK_NUM,869559.0,39.496413,23.092513,0.0,19.0,39.0,60.0,79.0,0,0.000000,0.000712
mainoccupationinc_384A,826347.0,24479.375708,11913.549853,2515.0,16140.0,22025.0,30012.0,174473.0,43212,4.969416,1.591207
credamount_770A,852195.0,16727.144127,8929.459423,1447.0,10528.0,14759.0,20689.0,150395.0,17364,1.996874,1.755873
annuity_780A,782942.0,1242.681145,662.241305,79.0,782.0,1097.0,1538.0,12069.0,86617,9.961026,1.746116
...,...,...,...,...,...,...,...,...,...,...,...
missing_tax_history,869559.0,0.250208,0.433133,0.0,0.0,0.0,1.0,1.0,0,0.000000,1.153419
missing_income,869559.0,0.049694,0.217312,0.0,0.0,0.0,0.0,1.0,0,0.000000,4.144327
missing_annuity,869559.0,0.099610,0.299480,0.0,0.0,0.0,0.0,1.0,0,0.000000,2.673907
missing_days_employed,869559.0,0.150563,0.357622,0.0,0.0,0.0,0.0,1.0,0,0.000000,1.954232


In [143]:
#Make sure there are no duplicates

print("Duplicate train case_ids:", X_train["case_id"].duplicated().sum())
print("Duplicate val case_ids:", X_val["case_id"].duplicated().sum())

Duplicate train case_ids: 0
Duplicate val case_ids: 0


In [144]:
# Check for infinite numerical values (Nitya)
inf_counts = pd.DataFrame({
    "positive_inf": np.isposinf(
        X_train[numeric_features]
    ).sum(),
    "negative_inf": np.isneginf(
        X_train[numeric_features]
    ).sum()
})

display(
    inf_counts[
        (inf_counts["positive_inf"] > 0) |
        (inf_counts["negative_inf"] > 0)
    ]
)

print(
    "Total infinite values:",
    np.isinf(
        X_train[numeric_features].to_numpy()
    ).sum()
)

,positive_inf,negative_inf


Total infinite values: 0


In [145]:
# Examine engineered ratio features (Nitya)
ratio_features = [
    "employment_age_ratio",
    "credit_income_ratio",
    "annuity_income_ratio",
    "annuity_credit_ratio",
    "tax_declared_income_ratio",
    "bureau_overdue_credit_ratio",
    "current_history_credit_ratio",
    "credit_vs_bureau_history",
    "credit_vs_prev_history"
]

ratio_features = [
    col for col in ratio_features
    if col in X_train.columns
]

display(X_train[ratio_features].describe().T)

,count,mean,std,min,25%,50%,75%,max
employment_age_ratio,738636.0,0.100554,0.072089,0.000000,0.049675,0.088873,0.137305,0.719737
credit_income_ratio,809827.0,0.844530,0.647079,0.022989,0.423915,0.670375,1.058755,20.641617
annuity_income_ratio,743965.0,0.062784,0.048218,0.002273,0.031476,0.049839,0.078829,1.338481
tax_declared_income_ratio,619608.0,1.198786,0.713795,0.062520,0.713679,1.031836,1.491804,15.988704
bureau_overdue_credit_ratio,576371.0,0.019712,0.016902,0.000000,0.008545,0.017119,0.026941,0.604605
current_history_credit_ratio,801140.0,0.507083,0.892601,0.009463,0.132453,0.243045,0.511226,48.884956
credit_vs_bureau_history,596508.0,1.234300,0.865360,0.036726,0.673674,1.015667,1.535441,40.811368


In [146]:
# Identify highly skewed numerical features (Nitya)
skewed_features = (
    numeric_summary[
        numeric_summary["skew"].abs() > 2
    ]
    .sort_values("skew", ascending=False)
)

display(skewed_features)

,count,mean,std,min,25%,50%,75%,max,missing,missing_pct,skew
current_history_credit_ratio,801140.0,0.507083,0.892601,0.009463,0.132453,0.243045,0.511226,48.884956,68419,7.868241,7.743364
missing_income,869559.0,0.049694,0.217312,0.000000,0.000000,0.000000,0.000000,1.000000,0,0.000000,4.144327
income_per_employed_year,701965.0,6673.636199,6569.947116,325.597132,2962.761506,4585.087623,7628.048369,124715.000000,167594,19.273448,3.346576
prev_credit_amount_min,695534.0,6616.413816,4442.122053,380.000000,3880.000000,5502.000000,7935.000000,174258.000000,174025,20.013018,3.146211
prior_credit_to_current,681693.0,0.748782,0.576311,0.020456,0.380822,0.597782,0.936921,21.036960,187866,21.604744,3.110480
credit_vs_bureau_history,596508.0,1.234300,0.865360,0.036726,0.673674,1.015667,1.535441,40.811368,273051,31.401090,3.078565
implied_term,767326.0,17.291945,13.966815,0.519103,8.345817,13.461142,21.686522,390.714286,102233,11.756879,2.954233
annuity_income_ratio,743965.0,0.062784,0.048218,0.002273,0.031476,0.049839,0.078829,1.338481,125594,14.443413,2.806463
credit_income_ratio,809827.0,0.844530,0.647079,0.022989,0.423915,0.670375,1.058755,20.641617,59732,6.869229,2.735896
missing_annuity,869559.0,0.099610,0.299480,0.000000,0.000000,0.000000,0.000000,1.000000,0,0.000000,2.673907


In [147]:
# Identify amount features (Nitya)
amount_features = [
    col for col in X_train.columns
    if col.endswith("A")
]

print("Amount features:", len(amount_features))
print(amount_features)

Amount features: 3
['mainoccupationinc_384A', 'credamount_770A', 'annuity_780A']


In [148]:
# Include engineered monetary features (Nitya)
engineered_amount_features = [
    "person_income_sum",
    "person_income_mean",
    "person_income_max",
    "tax_amount_sum",
    "tax_amount_mean",
    "tax_amount_max",
    "prev_credit_amount_sum",
    "prev_credit_amount_mean",
    "prev_credit_amount_max",
    "bureau_credit_amount_sum",
    "bureau_credit_amount_mean",
    "bureau_credit_amount_max",
    "bureau_overdue_amount_mean",
    "bureau_overdue_amount_max",
    "bureau_a2_overdue_sum",
    "bureau_a2_overdue_max",
    "bureau_a2_overdue_mean"
]

amount_features = [
    col
    for col in amount_features + engineered_amount_features
    if col in X_train.columns
]

print("Amount features to examine:", len(amount_features))

Amount features to examine: 20


In [149]:
# Examine extreme values in amount features (Nitya)
winsor_summary = pd.DataFrame({
    "1%": X_train[amount_features].quantile(0.01),
    "50%": X_train[amount_features].quantile(0.50),
    "99%": X_train[amount_features].quantile(0.99),
    "max": X_train[amount_features].max()
})

display(winsor_summary)

,1%,50%,99%,max
mainoccupationinc_384A,7521.000000,22025.000000,64421.540000,174473.0
credamount_770A,4606.000000,14759.000000,47329.060000,150395.0
annuity_780A,342.000000,1097.000000,3508.000000,12069.0
person_income_sum,8428.790000,32572.000000,92754.420000,230138.0
person_income_mean,8333.790000,22596.500000,59116.000000,193361.0
person_income_max,8394.790000,24977.000000,68558.000000,203989.0
tax_amount_sum,10025.000000,45352.000000,115408.130000,222542.0
tax_amount_mean,9796.000000,22770.333333,50695.043333,147877.0
tax_amount_max,9945.000000,26871.000000,64224.260000,169633.0
prev_credit_amount_sum,2804.000000,22095.000000,69104.670000,222374.0


In [150]:
# Fit winsorization thresholds (Nitya)
winsor_lower = X_train[amount_features].quantile(0.01)
winsor_upper = X_train[amount_features].quantile(0.99)

In [151]:
# Apply TRAIN fitted winsorization (Nitya)
for col in amount_features:
    X_train[col] = X_train[col].clip(
        lower=winsor_lower[col],
        upper=winsor_upper[col]
    )

    X_val[col] = X_val[col].clip(
        lower=winsor_lower[col],
        upper=winsor_upper[col]
    )

In [152]:
# Skewness of amount features (Nitya)
amount_skew = (
    X_train[amount_features]
    .skew()
    .sort_values(ascending=False)
)

display(amount_skew.to_frame("skew"))

,skew
prev_credit_amount_max,1.281051
credamount_770A,1.254165
annuity_780A,1.248115
prev_credit_amount_mean,1.222546
mainoccupationinc_384A,1.154659
person_income_mean,1.047340
person_income_max,1.044647
bureau_credit_amount_max,1.018605
bureau_credit_amount_mean,0.936738
bureau_a2_overdue_sum,0.884833


In [153]:
# Identify non-negative, highly skewed amount features (Nitya)
log_features = [
    col
    for col in amount_features
    if X_train[col].min(skipna=True) >= 0
    and abs(X_train[col].skew()) > 2
]

print("Log-transform candidates:")
print(log_features)

Log-transform candidates:
[]


In [154]:
# Apply log1p transformation (Nitya)
for col in log_features:
    X_train[col] = np.log1p(X_train[col])
    X_val[col] = np.log1p(X_val[col])


In [155]:
# Check skewness after log transformation (Nitya)
log_skew_check = (
    X_train[log_features]
    .skew()
    .sort_values(ascending=False)
)

display(log_skew_check.to_frame("skew_after_log1p"))

,skew_after_log1p


In [156]:
# Check transformations (Nitya)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)

print("Same columns:",
      X_train.columns.equals(X_val.columns))

print("Winsorized features:", len(amount_features))
print("Log-transformed features:", len(log_features))


Train shape: (869559, 85)
Validation shape: (130441, 85)
Same columns: True
Winsorized features: 20
Log-transformed features: 0


In [157]:
# Missingness Summary

missing_summary = pd.DataFrame({
    "missing_count": X_train.isna().sum(),
    "missing_pct": X_train.isna().mean() * 100
})

missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_pct", ascending=False)
)

display(missing_summary)

,missing_count,missing_pct
coapplicant_income_ratio,435373,50.068253
tax_amount_std,434753,49.996952
prev_credit_amount_std,348521,40.080202
max_overdue_to_income,321846,37.012555
bureau_a2_overdue_sum,312856,35.978697
...,...,...
person_income_sum,22779,2.619604
person_income_max,22779,2.619604
person_income_mean,22779,2.619604
person_income_min,22779,2.619604


In [158]:
# Examine missing values (Nitya)
missing_summary = pd.DataFrame({
    "missing_count": X_train.isna().sum(),
    "missing_pct": X_train.isna().mean() * 100
})

display(
    missing_summary[
        missing_summary["missing_count"] > 0
    ].sort_values(
        "missing_pct",
        ascending=False
    )
)

,missing_count,missing_pct
coapplicant_income_ratio,435373,50.068253
tax_amount_std,434753,49.996952
prev_credit_amount_std,348521,40.080202
max_overdue_to_income,321846,37.012555
bureau_a2_overdue_sum,312856,35.978697
...,...,...
person_income_sum,22779,2.619604
person_income_max,22779,2.619604
person_income_mean,22779,2.619604
person_income_min,22779,2.619604


In [159]:
# Fit encoder on TRAIN (Nitya)

# Identify categorical cols
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical columns to encode:", categorical_cols)

# Create encoder
ohe = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

ohe.fit(X_train[categorical_cols])

# Transform train and validation

X_train_ohe = ohe.transform(X_train[categorical_cols])
X_val_ohe = ohe.transform(X_val[categorical_cols])

encoded_cols = ohe.get_feature_names_out(categorical_cols)

# Convert encoded arrays to DataFrames
X_train_ohe = pd.DataFrame(
    X_train_ohe,
    columns=encoded_cols,
    index=X_train.index
)

X_val_ohe = pd.DataFrame(
    X_val_ohe,
    columns=encoded_cols,
    index=X_val.index
)

# Remove og categorical cols
X_train = X_train.drop(columns=categorical_cols)
X_val = X_val.drop(columns=categorical_cols)

# Add ohe cols
X_train = pd.concat([X_train, X_train_ohe], axis=1)
X_val = pd.concat([X_val, X_val_ohe], axis=1)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Same columns:", X_train.columns.equals(X_val.columns))

Categorical columns to encode: ['education_927M', 'maritalstatus_703M', 'description_5085714M', 'classificationofcontr_13M']
X_train: (869559, 104)
X_val: (130441, 104)
Same columns: True


In [160]:
from sklearn.impute import SimpleImputer

# Exclude identifiers and time columns
exclude_cols = ["case_id","WEEK_NUM"]

feature_cols = [
    col for col in X_train.select_dtypes(
        include=["number", "bool"]
    ).columns
    if col not in exclude_cols
    and col in X_val.columns
    and X_train[col].nunique(dropna=True) > 1
]
print("Number of features:", len(feature_cols))

Number of features: 98


In [161]:
# Check missing values before training
missing_counts = X_train[feature_cols].isna().sum()

print("Features with missing values:",
      (missing_counts > 0).sum())

print("Total missing values:",
      missing_counts.sum())

print("\nTop 15 features with missing values:")
print(missing_counts[missing_counts > 0]
      .sort_values(ascending=False)
      .head(15))

Features with missing values: 61
Total missing values: 13174965

Top 15 features with missing values:
coapplicant_income_ratio       435373
tax_amount_std                 434753
prev_credit_amount_std         348521
max_overdue_to_income          321846
pay_overdue_share              312856
bureau_a2_overdue_sum          312856
bureau_a2_overdue_max          312856
bureau_a2_overdue_mean         312856
bureau_overdue_amount_mean     293188
bureau_overdue_amount_max      293188
bureau_overdue_credit_ratio    293188
avg_overdue_per_contract       293188
overdue_contract_rate          293188
bureau_overdue_amount_sum      293188
riskassesment_940T             291393
dtype: int64


In [162]:
# Show all 61 features that have missing values
missing_cols = X_train[feature_cols].columns[
    X_train[feature_cols].isna().any()
].tolist()

for i, col in enumerate(missing_cols, 1):
    print(f"{i}. {col}")

1. mainoccupationinc_384A
2. credamount_770A
3. annuity_780A
4. days_employed_700P
5. riskassesment_940T
6. numberofqueries_146L
7. tax_record_count
8. tax_amount_sum
9. tax_amount_mean
10. tax_amount_max
11. person_income_sum
12. person_income_mean
13. person_income_max
14. employment_mean
15. employment_max
16. prev_application_count
17. prev_credit_amount_sum
18. prev_credit_amount_mean
19. prev_credit_amount_max
20. bureau_contract_count
21. bureau_credit_amount_sum
22. bureau_credit_amount_mean
23. bureau_credit_amount_max
24. bureau_overdue_amount_mean
25. bureau_overdue_amount_max
26. bureau_dpd_max
27. bureau_dpd_mean
28. bureau_a2_contract_count
29. bureau_a2_record_count
30. bureau_a2_overdue_sum
31. bureau_a2_overdue_max
32. bureau_a2_overdue_valid_count
33. bureau_a2_overdue_mean
34. pay_overdue_share
35. tax_amount_std
36. tax_amount_min
37. prev_credit_amount_std
38. prev_credit_amount_min
39. person_income_min
40. bureau_overdue_amount_sum
41. dpd_contract_rate
42. overd

In [163]:
numeric_missing = X_train[feature_cols].replace(
    [np.inf, -np.inf], np.nan
)

missing_cols = [
    c for c in feature_cols
    if numeric_missing[c].isna().any()
]

missing_diagnostics = pd.DataFrame({
    "missing_pct": (
        numeric_missing[missing_cols].isna().mean() * 100
    ).round(2),
    "non_missing_count": numeric_missing[missing_cols].count(),
    "skewness": numeric_missing[missing_cols].skew().round(2),
    "median": numeric_missing[missing_cols].median(),
    "mean": numeric_missing[missing_cols].mean(),
    "zero_pct_observed": (
        numeric_missing[missing_cols].eq(0).sum()
        / numeric_missing[missing_cols].count().replace(0, np.nan)
        * 100
    ).round(2)
})

display(
    missing_diagnostics.sort_values(
        "missing_pct", ascending=False
    )
)

,missing_pct,non_missing_count,skewness,median,mean,zero_pct_observed
coapplicant_income_ratio,50.07,434186,-0.22,0.685008,0.676136,0.00
tax_amount_std,50.00,434806,1.67,6680.592663,7993.832943,0.00
prev_credit_amount_std,40.08,521038,2.43,4002.787399,4988.613797,0.00
max_overdue_to_income,37.01,547713,1.87,0.022096,0.027491,9.73
pay_overdue_share,35.98,556703,-0.65,0.666667,0.660016,7.06
...,...,...,...,...,...,...
person_income_sum,2.62,846780,0.82,32572.000000,35732.695776,0.00
person_income_max,2.62,846780,1.04,24977.000000,27225.676189,0.00
person_income_mean,2.62,846780,1.05,22596.500000,24397.862315,0.00
person_income_min,2.62,846780,1.77,19419.000000,21657.776598,0.00


In [164]:
# Identify missingness patterns before imputation
raw_train = X_train[feature_cols].replace(
    [np.inf, -np.inf], np.nan
)

# Existing flags and coverage indicators
existing_flags = [
    "missing_prev_history",
    "missing_tax_history",
    "missing_income",
    "missing_annuity",
    "missing_days_employed",
    "missing_overdue",
    "has_prev",
    "has_tax",
    "is_thin_file",
    "has_coapplicant"
]

existing_flags = [
    c for c in existing_flags
    if c in X_train.columns
]

# Find distinct missingness patterns
seen_patterns = {}
new_indicators = []
already_covered = []
duplicate_patterns = []

for col in feature_cols:
    mask = raw_train[col].isna().astype("uint8")

    if mask.sum() == 0:
        continue

    # Skip patterns already represented by an existing flag
    covered = False

    for flag in existing_flags:
        old = X_train[flag]

        if old.notna().all():
            old = old.astype("uint8")

            if mask.equals(old) or mask.equals(1 - old):
                already_covered.append(col)
                covered = True
                break

    if covered:
        continue


    # Compare against patterns already selected
    is_duplicate = any(
        mask.equals(previous_mask)
        for previous_mask in seen_patterns.values()
    )

    if is_duplicate:
        duplicate_patterns.append(col)
        continue

    seen_patterns[col] = mask
    new_indicators.append(col)

print("Missing features already covered:",
      len(already_covered))

print("Duplicate missingness patterns:",
      len(duplicate_patterns))

print("New indicator candidates:",
      len(new_indicators))

print("\nNew indicator candidates:")
for col in new_indicators:
    print(col)


Missing features already covered: 30
Duplicate missingness patterns: 11
New indicator candidates: 20

New indicator candidates:
credamount_770A
riskassesment_940T
person_income_sum
employment_mean
bureau_a2_contract_count
bureau_a2_overdue_sum
tax_amount_std
prev_credit_amount_std
income_per_employed_year
coapplicant_income_ratio
credit_income_ratio
annuity_income_ratio
implied_term
tax_declared_income_ratio
bureau_exposure_ratio
prior_credit_to_current
max_overdue_to_income
prev_days_since_last_approval
current_history_credit_ratio
credit_vs_bureau_history


In [165]:
# Examine the new missingness indicator candidates

candidate_summary = pd.DataFrame({
    "feature": new_indicators,

    "missing_count": [
        X_train[col].isna().sum()
        for col in new_indicators
    ],

    "missing_pct": [
        round(X_train[col].isna().mean() * 100, 2)
        for col in new_indicators
    ]
})

candidate_summary = candidate_summary.sort_values(
    "missing_pct",
    ascending=False
)

display(candidate_summary)


,feature,missing_count,missing_pct
9,coapplicant_income_ratio,435373,50.07
6,tax_amount_std,434753,50.00
7,prev_credit_amount_std,348521,40.08
16,max_overdue_to_income,321846,37.01
5,bureau_a2_overdue_sum,312856,35.98
1,riskassesment_940T,291393,33.51
14,bureau_exposure_ratio,291038,33.47
4,bureau_a2_contract_count,286231,32.92
19,credit_vs_bureau_history,273051,31.40
13,tax_declared_income_ratio,249951,28.74


In [166]:
# ============================================================
# INVESTIGATE FEATURES WITH MORE THAN 20% MISSINGNESS
# ============================================================

# Use original data before imputation
raw = X_train.replace([np.inf, -np.inf], np.nan)

# Select features with more than 20% missingness
high_missing_cols = [
    col for col in new_indicators
    if raw[col].isna().mean() > 0.20
]

# Related features that may explain missingness
related_features = {
    "coapplicant_income_ratio": "has_coapplicant",
    "tax_amount_std": "tax_record_count",
    "prev_credit_amount_std": "prev_application_count",
    "max_overdue_to_income": "bureau_contract_count",
    "bureau_a2_overdue_sum": "bureau_a2_record_count",
    "riskassesment_940T": "is_thin_file",
    "bureau_exposure_ratio": "bureau_contract_count",
    "bureau_a2_contract_count": "bureau_a2_record_count",
    "credit_vs_bureau_history": "bureau_contract_count",
    "tax_declared_income_ratio": "tax_record_count",
    "prev_days_since_last_approval": "prev_application_count",
    "prior_credit_to_current": "prev_application_count"
}

# ============================================================
# 1. OVERALL MISSINGNESS SUMMARY
# ============================================================

summary = pd.DataFrame({
    "feature": high_missing_cols,
    "missing_count": [
        raw[col].isna().sum()
        for col in high_missing_cols
    ],
    "missing_pct": [
        round(raw[col].isna().mean() * 100, 2)
        for col in high_missing_cols
    ],
    "related_feature": [
        related_features.get(col)
        for col in high_missing_cols
    ]
}).sort_values("missing_pct", ascending=False)

print("Features above 20% missingness:", len(summary))
display(summary)

# ============================================================
# 2. INVESTIGATE MISSINGNESS BY RELATED FEATURE
# ============================================================

results = []

binary_features = [
    "has_coapplicant",
    "is_thin_file"
]

for feature in high_missing_cols:

    related = related_features.get(feature)

    if related not in raw.columns:
        continue

    # Group related values
    if related in binary_features:
        groups = raw[related].map({
            0: "0",
            1: "1"
        }).fillna("Missing")

    else:
        counts = raw[related]

        groups = pd.Series(
            np.select(
                [counts == 0, counts == 1, counts >= 2],
                ["0 records", "1 record", "2+ records"],
                default="Missing"
            ),index=raw.index
        )

    # Calculate missingness within each group
    for group in groups.unique():

        subset = raw.loc[groups == group, feature]

        results.append({
            "feature": feature,
            "related_feature": related,
            "group": group,
            "total": len(subset),
            "missing": subset.isna().sum(),
            "missing_pct": round(
                subset.isna().mean() * 100, 2
            )
        })

# Display all results together
group_summary = pd.DataFrame(results)

display(
    group_summary.sort_values(
        ["feature", "group"]
    )
)

Features above 20% missingness: 12


,feature,missing_count,missing_pct,related_feature
5,coapplicant_income_ratio,435373,50.07,has_coapplicant
3,tax_amount_std,434753,50.00,tax_record_count
4,prev_credit_amount_std,348521,40.08,prev_application_count
9,max_overdue_to_income,321846,37.01,bureau_contract_count
2,bureau_a2_overdue_sum,312856,35.98,bureau_a2_record_count
0,riskassesment_940T,291393,33.51,is_thin_file
7,bureau_exposure_ratio,291038,33.47,bureau_contract_count
1,bureau_a2_contract_count,286231,32.92,bureau_a2_record_count
11,credit_vs_bureau_history,273051,31.40,bureau_contract_count
6,tax_declared_income_ratio,249951,28.74,tax_record_count


,feature,related_feature,group,total,missing,missing_pct
3,bureau_a2_contract_count,bureau_a2_record_count,1 record,33768,0,0.00
2,bureau_a2_contract_count,bureau_a2_record_count,2+ records,549560,0,0.00
1,bureau_a2_contract_count,bureau_a2_record_count,Missing,286231,286231,100.00
6,bureau_a2_overdue_sum,bureau_a2_record_count,1 record,33768,13462,39.87
5,bureau_a2_overdue_sum,bureau_a2_record_count,2+ records,549560,13163,2.40
4,bureau_a2_overdue_sum,bureau_a2_record_count,Missing,286231,286231,100.00
20,bureau_exposure_ratio,bureau_contract_count,1 record,75714,3821,5.05
19,bureau_exposure_ratio,bureau_contract_count,2+ records,533025,26397,4.95
18,bureau_exposure_ratio,bureau_contract_count,Missing,260820,260820,100.00
14,coapplicant_income_ratio,has_coapplicant,0,434303,434303,100.00


In [167]:
# ============================================================
# ADD 8 NEW MISSINGNESS INDICATORS
# ============================================================

# 1. Select features that need missingness indicators
indicator_features = [
    "riskassesment_940T",
    "max_overdue_to_income",
    "bureau_a2_overdue_sum",
    "bureau_exposure_ratio",
    "credit_vs_bureau_history",
    "tax_declared_income_ratio",
    "prev_days_since_last_approval",
    "prior_credit_to_current"
]

# 2. Create indicators before imputation
# 1 = original value was missing
# 0 = original value was present

for col in indicator_features:
    indicator_name = "missing_" + col

    X_train[indicator_name] = (
        X_train[col]
        .replace([np.inf, -np.inf], np.nan)
        .isna()
        .astype("int8")
    )

    X_val[indicator_name] = (
        X_val[col]
        .replace([np.inf, -np.inf], np.nan)
        .isna()
        .astype("int8")
    )

# 3. Add the new indicators to feature_cols
new_indicator_cols = [
    "missing_" + col
    for col in indicator_features
]

feature_cols = list(dict.fromkeys(
    feature_cols + new_indicator_cols
))

# 4. Check updated dataset sizes
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

print("Total selected features:", len(feature_cols))
print("New indicators added:", len(new_indicator_cols))

X_train shape: (869559, 112)
X_val shape: (130441, 112)
Total selected features: 106
New indicators added: 8


In [168]:
# ============================================================
# NUMERICAL MISSING VALUE IMPUTATION
# ============================================================

from sklearn.impute import SimpleImputer

# 1. Create copies of the original features
X_train_ready = X_train[feature_cols].copy()
X_val_ready = X_val[feature_cols].copy()

# Replace infinite values with NaN
X_train_ready.replace(
    [np.inf, -np.inf], np.nan, inplace=True
)
X_val_ready.replace(
    [np.inf, -np.inf], np.nan, inplace=True
)

# ============================================================
# 2. STRUCTURAL ZERO IMPUTATION
# ============================================================

# Missing historical counts indicate no matching records
zero_fill_cols = [
    "tax_record_count",
    "prev_application_count",
    "bureau_contract_count",
    "bureau_a2_contract_count",
    "bureau_a2_record_count",
    "bureau_a2_overdue_valid_count"
]

zero_fill_cols = [
    col for col in zero_fill_cols
    if col in feature_cols
]

for col in zero_fill_cols:
    X_train_ready[col] = X_train_ready[col].fillna(0)
    X_val_ready[col] = X_val_ready[col].fillna(0)

# ============================================================
# 3. COAPPLICANT INCOME RATIO
# ============================================================

# No coapplicant: ratio is not applicable, use 0
# Coapplicant exists but ratio missing: use training median

ratio_col = "coapplicant_income_ratio"

ratio_median = X_train_ready.loc[
    X_train["has_coapplicant"] == 1,
    ratio_col
].median()


if not np.isfinite(ratio_median):
    ratio_median = 0

for original, ready in [
    (X_train, X_train_ready),
    (X_val, X_val_ready)
]:
    no_coapplicant = original["has_coapplicant"] == 0

    ready.loc[no_coapplicant, ratio_col] = 0
    ready[ratio_col] = ready[ratio_col].fillna(ratio_median)

# ============================================================
# 4. STANDARD DEVIATION FEATURES
# ============================================================

# Standard deviation is undefined with fewer than 2 records.
# Use 0 as a modeling convention in these cases.

std_features = [
    ("tax_amount_std", "tax_record_count"),
    ("prev_credit_amount_std", "prev_application_count")
]

for std_col, count_col in std_features:
    for original, ready in [
        (X_train, X_train_ready),
        (X_val, X_val_ready)
    ]:
        insufficient_records = (
            original[count_col].fillna(0) < 2
        )

        ready.loc[insufficient_records, std_col] = 0

# Handle any remaining missing standard deviation values
for std_col, count_col in std_features:
    fallback = X_train_ready[std_col].median()

    if not np.isfinite(fallback):
        fallback = 0

    X_train_ready[std_col] = X_train_ready[std_col].fillna(fallback)
    X_val_ready[std_col] = X_val_ready[std_col].fillna(fallback)

# ============================================================
# 5. MEDIAN IMPUTATION FOR REMAINING FEATURES
# ============================================================

special_cols = (
    zero_fill_cols
    + [ratio_col]
    + [col for col, count in std_features]
)

median_cols = [
    col for col in feature_cols
    if col not in special_cols
]

imputer = SimpleImputer(
    strategy="median",
    keep_empty_features=True
)

# Fit using training data only
X_train_ready[median_cols] = imputer.fit_transform(
    X_train_ready[median_cols]
)

# Apply training medians to validation
X_val_ready[median_cols] = imputer.transform(
    X_val_ready[median_cols]
)

# ============================================================
# 6. VERIFY RESULTS
# ============================================================

print("Train shape:", X_train_ready.shape)
print("Validation shape:", X_val_ready.shape)

print("Missing train:",
      X_train_ready.isna().sum().sum())

print("Missing validation:",
      X_val_ready.isna().sum().sum())

print("Infinite train:",
      np.isinf(X_train_ready.to_numpy()).sum())

print("Infinite validation:",
      np.isinf(X_val_ready.to_numpy()).sum())

print("\nImputation summary:")
print("Structural count features:", len(zero_fill_cols))
print("Coapplicant ratio features:", 1)
print("Standard deviation features:", len(std_features))
print("Median-imputed feature columns:", len(median_cols))


Train shape: (869559, 106)
Validation shape: (130441, 106)
Missing train: 0
Missing validation: 0
Infinite train: 0
Infinite validation: 0

Imputation summary:
Structural count features: 6
Coapplicant ratio features: 1
Standard deviation features: 2
Median-imputed feature columns: 97


In [174]:
# Align target indices with the processed feature matrices
y_train = y_train.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)

# Align final feature matrix indices
X_train_ready = X_train_ready.reset_index(drop=True)
X_val_ready = X_val_ready.reset_index(drop=True)

In [175]:
# Final validation checks
assert X_train_ready.columns.equals(X_val_ready.columns)

assert len(X_train_ready) == len(y_train)
assert len(X_val_ready) == len(y_val)

assert not X_train_ready.isna().any().any()
assert not X_val_ready.isna().any().any()

assert np.isfinite(X_train_ready.to_numpy()).all()
assert np.isfinite(X_val_ready.to_numpy()).all()

assert all(col in X_train_ready.columns for col in new_indicator_cols)
assert all(col in X_val_ready.columns for col in new_indicator_cols)

print("All preprocessing checks passed!")

All preprocessing checks passed!


In [177]:
# Verify that feature rows match their target labels
assert X_train_ready.index.equals(y_train.index), \
    "Training feature and label indices differ"

assert X_val_ready.index.equals(y_val.index), \
    "Validation feature and label indices differ"

print("Feature and label alignment checks passed!")

Feature and label alignment checks passed!


**Handling Missing Values**

Before using the dataset for modeling, I checked which numerical features had missing values and how much data was missing. I found 61 features with missing values out of the original 98 selected features.

I also looked at the percentage of missing values in each feature. I focused more on features with over 20% missingness because a large amount of their data was missing. I compared these features with related columns, such as the number of previous applications, tax records, and credit bureau records, to better understand why the values were missing.

**Filling Missing Values**

I used different methods to fill missing values depending on what the feature represented.

Zero imputation: For six features that counted historical records, I filled missing values with 0 because they generally represented applicants with no matching records in those datasets.

Coapplicant income ratio: If an applicant did not have a coapplicant, I filled the missing ratio with 0 because the ratio did not apply to them. If they had a coapplicant but the ratio was missing, I used the median from the training data.

Standard deviation features: For tax_amount_std and prev_credit_amount_std, I noticed that values were missing when applicants had fewer than two records. Since standard deviation cannot be calculated from only one observation using the default method, I filled these values with 0 as a placeholder.

Median imputation: For the remaining numerical features, I used the median because it is less affected by unusually high or low values compared to the mean.

I calculated the median values using only the training dataset and then applied those same values to the validation dataset. This helps prevent information from the validation data from influencing preprocessing.

**Adding Missingness Indicators**

I also wanted to keep track of which values were originally missing, since missing information might be useful for predicting loan defaults.

I compared the missingness patterns across features and checked whether existing indicators already captured the same information. After this analysis, I selected 8 additional missingness indicators.

Each indicator is a new column containing:

1: The original feature value was missing.

0: The original feature value was available.

This allows the model to recognize when information was originally missing, even after I filled the missing values.

After adding these indicators, the number of selected numerical features increased from 98 to 106.